# 03 — Feature Engineering + Baseline ML
## Dataset: Fraud detection 1M transactions

**Mục tiêu Phase 1**:
- Tạo feature bổ sung (đặc biệt graph-related)
- Train 3 baseline models: Logistic Regression → Random Forest → XGBoost/LightGBM
- Đánh giá đúng metrics cho fraud detection (PR-AUC, Recall, FPR…)
- Chọn model tốt nhất làm baseline production

## 0. Setup

In [3]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve,
    precision_score, recall_score, f1_score
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries loaded.")

Libraries loaded.


In [5]:
DATA_DIR = Path("../data/processed/fraud_1m_processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load dữ liệu đã clean
tx = pd.read_parquet(DATA_DIR / "transactions_clean.parquet")
edges = pd.read_parquet(DATA_DIR / "network_edges_clean.parquet")
accounts = pd.read_parquet(DATA_DIR / "account_profiles_clean.parquet")

print("transactions :", tx.shape)
print("network_edges:", edges.shape)
print("accounts     :", accounts.shape)
print("\nFraud rate  :", round(tx["is_fraud"].mean()*100, 3), "%")

transactions : (1000000, 23)
network_edges: (7411, 6)
accounts     : (50000, 23)

Fraud rate  : 1.714 %


## 1. Feature Engineering

### 1.1 Feature từ thời gian + flag

In [6]:
# is_night (tín hiệu mạnh từ EDA)
tx["is_night"] = tx["hour_of_day"].isin([0, 1, 2, 3, 4, 5]).astype("int8")

# High flags
tx["velocity_high"] = (tx["velocity_1h"] >= 5).astype("int8")
tx["ip_risk_high"] = (tx["ip_risk_score"] >= 50).astype("int8")
tx["amount_ratio_high"] = (tx["amount_vs_avg_ratio"] >= 5).astype("int8")

# Log amount
tx["amount_log"] = np.log1p(tx["amount"])

print("Đã tạo: is_night, velocity_high, ip_risk_high, amount_ratio_high, amount_log")

Đã tạo: is_night, velocity_high, ip_risk_high, amount_ratio_high, amount_log


### 1.2 Feature từ Graph (quan trọng)

In [7]:
# 1. in_ring
ring_accounts = set(edges.dropna(subset=["ring_id"])["account_a"]) | \
                set(edges.dropna(subset=["ring_id"])["account_b"])
tx["in_ring"] = tx["account_id"].isin(ring_accounts).astype("int8")

print(f"Số account trong ring: {len(ring_accounts):,}")
print(tx.groupby("in_ring")["is_fraud"].mean())

Số account trong ring: 6,783
in_ring
0    0.016156
1    0.022813
Name: is_fraud, dtype: float64


In [8]:
# 2. account_degree (số kết nối)
degree = pd.concat([edges["account_a"], edges["account_b"]]).value_counts()
tx["account_degree"] = tx["account_id"].map(degree).fillna(0).astype("int16")

print(tx["account_degree"].describe())

count    1000000.000000
mean           0.378203
std            1.476557
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max           21.000000
Name: account_degree, dtype: float64


In [9]:
# 3. shared_type count (số loại liên kết khác nhau của account)
# Tùy chọn - có thể bỏ nếu muốn đơn giản
shared_counts = edges.groupby("account_a")["shared_type"].nunique()
shared_counts = shared_counts.add(
    edges.groupby("account_b")["shared_type"].nunique(), fill_value=0
).astype(int)

tx["n_shared_types"] = tx["account_id"].map(shared_counts).fillna(0).astype("int8")
print(tx["n_shared_types"].value_counts().sort_index())

n_shared_types
0    851794
1    107340
2      7959
3      7303
4     10924
5      7528
6      5052
7      1384
8       716
Name: count, dtype: int64


### 1.3 Join thêm thông tin từ account_profiles

In [10]:
profile_cols = [
    "account_id", "risk_score", "is_high_risk", "fraud_rate",
    "avg_velocity", "pct_foreign", "is_fraudster", "avg_amount"
]
profile_cols = [c for c in profile_cols if c in accounts.columns]

tx = tx.merge(accounts[profile_cols], on="account_id", how="left", suffixes=("", "_profile"))

# Điền missing (nếu có)
for col in ["risk_score", "fraud_rate", "avg_velocity", "pct_foreign", "avg_amount"]:
    if col in tx.columns:
        tx[col] = tx[col].fillna(tx[col].median())

for col in ["is_high_risk", "is_fraudster"]:
    if col in tx.columns:
        tx[col] = tx[col].fillna(0).astype("int8")

print("Đã join account_profiles. Shape hiện tại:", tx.shape)

Đã join account_profiles. Shape hiện tại: (1000000, 38)


### 1.4 Chọn danh sách feature cuối cùng

In [12]:
FEATURE_COLS = [
    # Số mạnh nhất từ EDA
    "velocity_1h",
    "ip_risk_score",
    "amount",
    "amount_vs_avg_ratio",
    "amount_log",
    "time_since_last_s",
    "account_age_days",
    "credit_limit",

    # Thời gian
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "is_night",

    # Flag
    "is_foreign_txn",
    "card_present",
    "device_known",
    "has_2fa",
    "velocity_high",
    "ip_risk_high",
    "amount_ratio_high",

    # Graph
    "in_ring",
    "account_degree",
    "n_shared_types",

    # Từ account_profiles
    "risk_score",
    "is_high_risk",
    "fraud_rate",
    "avg_velocity",
    "pct_foreign",
    "is_fraudster",
]

# Chỉ giữ cột thực sự tồn tại
FEATURE_COLS = [c for c in FEATURE_COLS if c in tx.columns]
print(f"Số feature sử dụng: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

Số feature sử dụng: 28
['velocity_1h', 'ip_risk_score', 'amount', 'amount_vs_avg_ratio', 'amount_log', 'time_since_last_s', 'account_age_days', 'credit_limit', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'is_foreign_txn', 'card_present', 'device_known', 'has_2fa', 'velocity_high', 'ip_risk_high', 'amount_ratio_high', 'in_ring', 'account_degree', 'n_shared_types', 'risk_score', 'is_high_risk', 'fraud_rate', 'avg_velocity', 'pct_foreign', 'is_fraudster']


In [13]:
# Kiểm tra missing trong feature
print(tx[FEATURE_COLS].isnull().sum().sort_values(ascending=False).head(10))

velocity_1h            0
ip_risk_score          0
amount                 0
amount_vs_avg_ratio    0
amount_log             0
time_since_last_s      0
account_age_days       0
credit_limit           0
hour_of_day            0
day_of_week            0
dtype: int64


## 2. Train / Validation Split

In [14]:
X = tx[FEATURE_COLS].copy()
y = tx["is_fraud"].copy()

# Điền missing còn sót (nếu có)
X = X.fillna(X.median(numeric_only=True))

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Fraud rate:", round(y_train.mean()*100, 3), "%")
print("Val  :", X_val.shape,   "Fraud rate:", round(y_val.mean()*100, 3), "%")

Train: (800000, 28) Fraud rate: 1.714 %
Val  : (200000, 28) Fraud rate: 1.715 %


## 3. Hàm đánh giá chung

In [15]:
def evaluate_model(model, X_val, y_val, model_name="Model", threshold=0.5):
    """Đánh giá đầy đủ metrics cho fraud detection"""
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_prob = model.decision_function(X_val)
        y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min() + 1e-8)

    y_pred = (y_prob >= threshold).astype(int)

    roc_auc = roc_auc_score(y_val, y_prob)
    pr_auc = average_precision_score(y_val, y_prob)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    print(f"\n{'='*50}")
    print(f" {model_name}")
    print(f"{'='*50}")
    print(f"ROC-AUC     : {roc_auc:.4f}")
    print(f"PR-AUC      : {pr_auc:.4f}   ← quan trọng nhất")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"FPR         : {fpr:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"TN={tn:,}  FP={fp:,}")
    print(f"FN={fn:,}  TP={tp:,}")

    return {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "y_prob": y_prob
    }

## 4. Model 1 — Logistic Regression (baseline đơn giản)

In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",   # quan trọng vì mất cân bằng
    random_state=42,
    n_jobs=-1
)
lr.fit(X_train_scaled, y_train)

res_lr = evaluate_model(lr, X_val_scaled, y_val, "Logistic Regression")


 Logistic Regression
ROC-AUC     : 0.9968
PR-AUC      : 0.8844   ← quan trọng nhất
Precision   : 0.3625
Recall      : 0.9787
F1-score    : 0.5291
FPR         : 0.0300

Confusion Matrix:
TN=190,670  FP=5,901
FN=73  TP=3,356


## 5. Model 2 — Random Forest

In [17]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

res_rf = evaluate_model(rf, X_val, y_val, "Random Forest")


 Random Forest
ROC-AUC     : 0.9976
PR-AUC      : 0.9078   ← quan trọng nhất
Precision   : 0.4040
Recall      : 0.9840
F1-score    : 0.5728
FPR         : 0.0253

Confusion Matrix:
TN=191,593  FP=4,978
FN=55  TP=3,374


## 6. Model 3 — XGBoost / LightGBM (mục tiêu chính)

In [18]:
# Thử XGBoost trước, nếu không có thì dùng LightGBM
try:
    from xgboost import XGBClassifier
    USE_XGB = True
except ImportError:
    USE_XGB = False
    print("xgboost chưa cài → thử lightgbm")

if not USE_XGB:
    try:
        from lightgbm import LGBMClassifier
        USE_LGBM = True
    except ImportError:
        USE_LGBM = False
        print("Cần cài xgboost hoặc lightgbm: pip install xgboost lightgbm")

In [ ]:

if USE_XGB:
    # scale_pos_weight = số negative / số positive
    spw = (y_train == 0).sum() / (y_train == 1).sum()

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=30
    )
    
    xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )
    res_xgb = evaluate_model(xgb, X_val, y_val, "XGBoost")

elif USE_LGBM:
    lgbm = LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    lgbm.fit(X_train, y_train)
    res_xgb = evaluate_model(lgbm, X_val, y_val, "LightGBM")
else:
    res_xgb = None
    print("Bỏ qua model boosting vì chưa cài thư viện")

[0]	validation_0-aucpr:0.64917
[50]	validation_0-aucpr:0.89870
[100]	validation_0-aucpr:0.91331
[150]	validation_0-aucpr:0.92185
[200]	validation_0-aucpr:0.92372
[250]	validation_0-aucpr:0.92396
[275]	validation_0-aucpr:0.92378

 XGBoost
ROC-AUC     : 0.9980
PR-AUC      : 0.9241   ← quan trọng nhất
Precision   : 0.4471
Recall      : 0.9854
F1-score    : 0.6151
FPR         : 0.0213

Confusion Matrix:
TN=192,392  FP=4,179
FN=50  TP=3,379


In [21]:
imp = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(imp.head(15))

is_fraudster         0.424044
ip_risk_score        0.205634
ip_risk_high         0.084312
fraud_rate           0.064550
velocity_high        0.057016
velocity_1h          0.053619
is_foreign_txn       0.032479
device_known         0.026782
amount_ratio_high    0.008961
amount               0.006861
is_night             0.005740
card_present         0.004264
amount_log           0.004042
avg_velocity         0.003210
hour_of_day          0.002662
dtype: float32


In [22]:
LEAKY_CANDIDATES = ["fraud_rate", "is_fraudster", "risk_score", "is_high_risk"]

FEATURE_COLS_CLEAN = [c for c in FEATURE_COLS if c not in LEAKY_CANDIDATES]

X_clean = tx[FEATURE_COLS_CLEAN].fillna(tx[FEATURE_COLS_CLEAN].median(numeric_only=True))
X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

xgb_clean = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train_c==0).sum()/(y_train_c==1).sum(),
    eval_metric="aucpr", random_state=42, n_jobs=-1,
    early_stopping_rounds=30
)
xgb_clean.fit(X_train_c, y_train_c, eval_set=[(X_val_c, y_val_c)], verbose=50)

res_clean = evaluate_model(xgb_clean, X_val_c, y_val_c, "XGBoost (no leaky features)")

[0]	validation_0-aucpr:0.71992
[50]	validation_0-aucpr:0.82524
[100]	validation_0-aucpr:0.83725
[150]	validation_0-aucpr:0.84235
[200]	validation_0-aucpr:0.84270
[223]	validation_0-aucpr:0.84244

 XGBoost (no leaky features)
ROC-AUC     : 0.9937
PR-AUC      : 0.8430   ← quan trọng nhất
Precision   : 0.2837
Recall      : 0.9635
F1-score    : 0.4383
FPR         : 0.0424

Confusion Matrix:
TN=188,227  FP=8,344
FN=125  TP=3,304


## 7. So sánh tổng hợp

In [20]:
results = [res_lr, res_rf]
if res_xgb is not None:
    results.append(res_xgb)

df_results = pd.DataFrame(results)[["model", "roc_auc", "pr_auc", "precision", "recall", "f1", "fpr"]]
df_results = df_results.sort_values("pr_auc", ascending=False)

print("\n=== BẢNG SO SÁNH (sắp xếp theo PR-AUC) ===")
display(df_results.round(4))


=== BẢNG SO SÁNH (sắp xếp theo PR-AUC) ===


,model,roc_auc,pr_auc,precision,recall,f1,fpr
2,XGBoost,0.9980,0.9241,0.4471,0.9854,0.6151,0.0213
1,Random Forest,0.9976,0.9078,0.4040,0.9840,0.5728,0.0253
0,Logistic Regression,0.9968,0.8844,0.3625,0.9787,0.5291,0.0300


In [ ]:
# Vẽ PR Curve
plt.figure(figsize=(8, 6))
for res in results:
    precision, recall, _ = precision_recall_curve(y_val, res["y_prob"])
    plt.plot(recall, precision, label=f"{res['model']} (PR-AUC={res['pr_auc']:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Feature Importance (model tốt nhất)

In [ ]:
best_model = None
best_name = ""

if res_xgb is not None and res_xgb["pr_auc"] >= res_rf["pr_auc"]:
    best_model = xgb if USE_XGB else lgbm
    best_name = "XGBoost" if USE_XGB else "LightGBM"
else:
    best_model = rf
    best_name = "Random Forest"

print(f"Best model: {best_name}")

if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    imp = imp.sort_values(ascending=False)

    plt.figure(figsize=(10, 7))
    imp.head(20).plot(kind="barh")
    plt.title(f"Feature Importance — {best_name}")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    print("\nTop 15 features:")
    print(imp.head(15))

## 9. Lưu model tốt nhất (tùy chọn)

In [ ]:
import joblib

# Uncomment khi muốn lưu
# joblib.dump(best_model, MODEL_DIR / f"baseline_{best_name.lower().replace(' ', '_')}.pkl")
# joblib.dump(FEATURE_COLS, MODEL_DIR / "feature_cols.pkl")
# print("Đã lưu model và feature list")

## 10. Kết luận Phase 1

Sau khi chạy xong, hãy ghi lại:

1. Model nào có **PR-AUC** cao nhất?
2. Recall và FPR của model đó là bao nhiêu?
3. Top 5 feature quan trọng nhất là gì?
4. `in_ring` và `account_degree` có nằm trong top không? (kiểm tra giá trị của Graph feature)

**Next steps**:
- Nếu hài lòng với baseline → sang Phase 2 (Anomaly Detection) hoặc Phase 3 (Rule Engine)
- Có thể tune threshold để cân bằng Precision/Recall theo nhu cầu business